In [1]:
!unzip archive\ \(10\).zip

Archive:  archive (10).zip
  inflating: loan_prediction.csv     


In [3]:
import pandas as pd

# Load the unzipped file
df = pd.read_csv('loan_prediction.csv')

# Let's look at the columns and their data types
print("--- Dataset Info ---")
print(df.info())
print("\n--- First 5 Rows ---")
display(df.head())

--- Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 67.2+ KB
None

--- First 5 Rows ---


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [4]:
# 1. Handling Missing Values
# Fill categorical missing values with the most frequent value (Mode)
for col in ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Credit_History']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill numerical missing values with the Median (safer than Mean for income)
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median())

# 2. Label Encoding (Converting categorical text to numbers)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

# List of columns to encode
cat_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status', 'Dependents']

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# 3. Drop 'Loan_ID' as it's just a label and not useful for prediction
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

print("Preprocessing Complete! Here is the numerical data:")
df.head()

Preprocessing Complete! Here is the numerical data:


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,1,0,0,0,0,5849,0.0,128.0,360.0,1.0,2,1
1,1,1,1,0,0,4583,1508.0,128.0,360.0,1.0,0,0
2,1,1,0,0,1,3000,0.0,66.0,360.0,1.0,2,1
3,1,1,0,1,0,2583,2358.0,120.0,360.0,1.0,2,1
4,1,0,0,0,0,6000,0.0,141.0,360.0,1.0,2,1


In [5]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# 1. Split features (X) and target (y)
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# 2. Split into Training and Testing sets (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Handle Class Imbalance using SMOTE
# This creates synthetic data points for the minority class
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 4. Feature Scaling
# Since income and loan amounts are much larger than gender (0 or 1),
# we scale them so the model treats them fairly.
scaler = StandardScaler()
X_train_res = scaler.fit_transform(X_train_res)
X_test = scaler.transform(X_test)

print(f"Original training shape: {X_train.shape}")
print(f"Resampled training shape (after SMOTE): {X_train_res.shape}")
print("Balance of classes in training set:", np.bincount(y_train_res))

Original training shape: (491, 11)
Resampled training shape (after SMOTE): (684, 11)
Balance of classes in training set: [342 342]


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Initialize Models
log_model = LogisticRegression()
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Train Models
log_model.fit(X_train_res, y_train_res)
rf_model.fit(X_train_res, y_train_res)

# 3. Predictions
log_pred = log_model.predict(X_test)
rf_pred = rf_model.predict(X_test)

# 4. Evaluation
print("--- Logistic Regression Report ---")
print(classification_report(y_test, log_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, log_model.predict_proba(X_test)[:, 1]):.2f}")

print("\n--- Random Forest Report ---")
print(classification_report(y_test, rf_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1]):.2f}")

--- Logistic Regression Report ---
              precision    recall  f1-score   support

           0       0.71      0.51      0.59        43
           1       0.77      0.89      0.83        80

    accuracy                           0.76       123
   macro avg       0.74      0.70      0.71       123
weighted avg       0.75      0.76      0.74       123

ROC-AUC: 0.71

--- Random Forest Report ---
              precision    recall  f1-score   support

           0       0.69      0.47      0.56        43
           1       0.76      0.89      0.82        80

    accuracy                           0.74       123
   macro avg       0.72      0.68      0.69       123
weighted avg       0.73      0.74      0.73       123

ROC-AUC: 0.76
